In [1]:
#from mace.calculators import mace_off 
from smilesbox.smilesbox import SMILESbox
from chgnet.model.dynamics import CHGNetCalculator 

## 1. build the cell (Forward)

In [2]:
#smiles = ["N#N","O=O","O=O"]
#smiles = ["N(=O)[O]","N(=O)[O]"]
#smiles = ["C(=O)=O","[HH]"] # forward 
#smiles = ["[HH]","[HH]","O=O"]
#smiles = ["[N]=O","[N]=O","O=O"]
smiles = ["O=S=O","[O-][N+]=O","O"]
sm = SMILESbox(
    smiles = smiles * 4,
    box_size=20,
    min_distance=1.0
)
atoms = sm.atoms

In [3]:
calculator = CHGNetCalculator(use_device="mps", on_isolated_atoms="ignore")
#calculator = mace_off(model="small")
#calculator = EMT()

CHGNet v0.3.0 initialized with 412,525 parameters
CHGNet will run on mps


In [ ]:
from ase.io import read
from claude_code.nano_reactor import NanoReactor
import warnings 
warnings.simplefilter("ignore")
output_dir = "nanoreactor_forward"
reactor = NanoReactor(
    atoms        = atoms,
    calculator   = calculator,
    temperature  = 4500,          # K
    total_time_fs= 10000,       # 100 ps
    timestep_fs  = 0.25,
    tau_min_fs   = 10,
    reactants    = {"SO2","NO2","H2O"},
    products     = {"H2SO4","NO"},
    output_dir   = output_dir,
)

reactor.run()
reactor.report()
reactor.save_summary_csv(f"{output_dir}/results.csv")
reactor.plot()


NanoReactor starting
  36 atoms | T=4500 K | NVT
  40000 steps × 0.25 fs = 10.0 ps
  τ_min = 10 fs | BO cutoffs = defaults (0.3)
  Output → nanoreactor_forward

Running MD...
  [SpeciesTracker] processed 500 frames, 3 unique formulae seen
  [SpeciesTracker] processed 1000 frames, 5 unique formulae seen
  [SpeciesTracker] processed 1500 frames, 5 unique formulae seen
  [SpeciesTracker] processed 2000 frames, 7 unique formulae seen
  [SpeciesTracker] processed 2500 frames, 7 unique formulae seen
  [SpeciesTracker] processed 3000 frames, 7 unique formulae seen
  [SpeciesTracker] processed 3500 frames, 7 unique formulae seen
  [SpeciesTracker] processed 4000 frames, 7 unique formulae seen
  [SpeciesTracker] processed 4500 frames, 7 unique formulae seen
  [SpeciesTracker] processed 5000 frames, 9 unique formulae seen
  [SpeciesTracker] processed 5500 frames, 9 unique formulae seen
  [SpeciesTracker] processed 6000 frames, 10 unique formulae seen
  [SpeciesTracker] processed 6500 frames, 11

In [ ]:
from claude_code.pathway_enumerator import PathwayEnumerator

# Pull observed transitions from your MD reaction network
observed_edges = {(src, tgt) for src, tgt, _ in reactor.network.get_neb_pairs()}

enumerator = PathwayEnumerator(
    overall_reaction = "2NO + O2 = 2NO2",
    species_pool     = ["NO", "O2", "NO2", "NO3", "N2O2", "ONOO"],
    observed_edges   = observed_edges,   # filters to MD-supported steps only
    max_path_steps   = 4,
    max_paths        = 20,
)

pathways = enumerator.enumerate()
enumerator.print_pathways(pathways)

# Feed unique steps directly to NEBRunner
neb_jobs = enumerator.to_neb_steps(pathways)